# Aggregation Strategy 2: Attention Pool (VARS default)

Learns per-view importance weights, but the views themselves do not interact. This is the aggregation block used in the original VARS paper and the one that produced the 39.97 test LB result.

## Formulation

Given per-view features $V \in \mathbb{R}^{B \times N \times d}$ (batch, views, feature dim):

1. Project: $A = V \cdot W$, with $W \in \mathbb{R}^{d \times d}$ (learnable).
2. Pairwise view similarity: $S = \text{ReLU}(A A^\top) \in \mathbb{R}^{B \times N \times N}$.
3. Normalize each row, then sum across columns to get per-view weights $w \in \mathbb{R}^{B \times N}$.
4. Weighted sum: $z = \sum_{i=1}^{N} w_i \cdot A_i$.

## Properties

| Property | Value |
|---|---|
| Learnable parameters | $d \times d \approx 262\,\text{k}$ for $d=512$ |
| Views interact? | **No** — each view gets a scalar weight based on pairwise similarity, but is never updated with information from other views |
| Permutation-invariant over views | Yes |
| Handles variable view count | Yes |

## What this isolates

Compared to mean pool, attention pool adds exactly one capability: **learned per-view weighting**. If attention pool beats mean pool significantly, the gain is attributable to learned weighting. If not, every view is already roughly equally informative.

## Code

Implemented as `WeightedAggregate` in [`VARS model/mvaggregate.py`](../VARS%20model/mvaggregate.py).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../VARS model'))

import torch
from torch import nn
from mvaggregate import WeightedAggregate

print('WeightedAggregate imported from:', WeightedAggregate.__module__)

/venv/vars/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WeightedAggregate imported from: mvaggregate


## Forward-pass demo on the inner aggregation logic

`WeightedAggregate.forward` wraps the full pipeline (backbone → aggregation). To see what the aggregation itself does, we reproduce its math on pre-extracted view features.

In [2]:
torch.manual_seed(0)
B, V, d = 2, 3, 512
view_features = torch.randn(B, V, d)

# Learnable projection (matches self.attention_weights in the real class)
W = nn.Parameter(torch.rand(d, d) * 2 - 1)

A = view_features @ W                         # (B, V, d)
S = torch.relu(A @ A.transpose(1, 2))         # (B, V, V) pairwise similarity

# Normalize each (view x view) matrix, then sum rows to get per-view weight.
S_flat = S.reshape(B, V * V)
w = S_flat / S_flat.sum(dim=1, keepdim=True)  # (B, V*V)
w = w.reshape(B, V, V).sum(dim=1)             # (B, V)

z = (A * w.unsqueeze(-1)).sum(dim=1)          # (B, d)

print('Per-view features:', view_features.shape)
print('Per-view weights: ', w.shape, '→', w[0].tolist())
print('Pooled feature:   ', z.shape)

Per-view features: torch.Size([2, 3, 512])
Per-view weights:  torch.Size([2, 3]) → [0.32518211007118225, 0.33477434515953064, 0.3400435149669647]
Pooled feature:    torch.Size([2, 512])


### Key observation

The weights $w$ are produced *from* the views but are applied *to* the same views without mixing information across them. Each output dimension of $z$ is a linear combination of the corresponding dimensions of $A_1, \dots, A_V$. That is why we say attention pool **does not make views interact** — it only learns how to weigh them.

In [3]:
# Parameter count of the aggregation block alone
class IdentityBackbone(nn.Module):
    def forward(self, x):
        return x

agg = WeightedAggregate(model=IdentityBackbone(), feat_dim=512)
own_params = sum(p.numel() for name, p in agg.named_parameters() if not name.startswith('model.'))
print(f'Attention pool aggregation block parameters: {own_params:,}')

Attention pool aggregation block parameters: 263,168
